# Quant Finance from Zero — A Hands-On Notebook

Welcome. This notebook teaches you the mathematics and programming behind quantitative finance, **assuming you have never seen any of it before**. Every concept is introduced in plain language, then immediately turned into runnable Python so you can *see* the idea move.

The philosophy here (inspired by the structure of [QuantFrame](https://quantframe.io)) is simple: **a quant is someone who answers questions about money using math and code.** So we never learn math for its own sake — every tool we pick up is wielded against a trading or risk question.

## The journey

| Part | Topic | The trading question it answers |
|------|-------|---------------------------------|
| 0 | **Programming Foundations** | How do I make a computer do the arithmetic for me, thousands of times? |
| 1 | **Calculus** | How fast is a price changing, and where is risk maximized? |
| 2 | **Linear Algebra** | How do I handle a whole *portfolio* of assets at once? |
| 3 | **Probability & Statistics** | What is my expected profit, and how uncertain is it? |
| 4 | **Stochastic Calculus** | How do prices move randomly through time, and what is an option worth? |
| 5 | **Capstone Projects** | Put it all together: Monte Carlo, portfolio optimizer, volatility surface |

## How to use this notebook

1. **Read every markdown cell**, then **run the code cell beneath it** (`Shift+Enter`).
2. **Change the numbers** and re-run. The fastest way to build intuition is to break things and see what happens.
3. Each section ends with a **"Your turn"** exercise. Try it before moving on.
4. Math is written so you can read it aloud. Where you see a symbol like $\sigma$ (sigma), I'll always say what it means in words first.

Let's begin.


---
## Setup — run this first

This installs/imports the libraries we use throughout. If you're on a fresh machine, the `pip install` line gets everything you need. On most data-science setups (Anaconda) these are already present.


In [ ]:
# If anything is missing, uncomment the next line and run it once:
# !pip install numpy pandas matplotlib scipy

import numpy as np                 # the workhorse for numbers and arrays
import pandas as pd                # tables of data (like a spreadsheet in code)
import matplotlib.pyplot as plt    # plotting / charts
from scipy import stats            # statistics helpers

# Make plots look clean and reproducible
np.random.seed(42)                 # fixes randomness so your results match mine
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Environment ready. NumPy version:", np.__version__)

---
# Part 0 — Programming Foundations

**The trading question:** *How do I get a computer to do arithmetic for me — not once, but a million times, on real market data?*

You cannot do modern quant finance by hand. A single options desk reprices thousands of instruments per second. So before any math, we learn just enough Python to express ideas as code. We'll cover: variables, arrays (the single most important object in quant work), functions, and plotting.

If you already know Python, skim this — but **do run the cells**, because the rest of the notebook builds directly on these objects.


### 0.1 Variables and arithmetic

A **variable** is a named box that holds a value. Think of it as a label you stick on a number so you can refer to it later.


In [ ]:
# A simple trade: buy 100 shares at $42.50, sell at $45.10
shares = 100
buy_price = 42.50
sell_price = 45.10

profit = shares * (sell_price - buy_price)   # P&L = quantity x price change
print("Profit on the trade: $", profit)

# Returns are usually more useful than dollar profit, because they're comparable across assets.
# Return = (end - start) / start
ret = (sell_price - buy_price) / buy_price
print("Return:", round(ret * 100, 3), "%")

### 0.2 Arrays — the heartbeat of quant finance

A single price is one number. But a *price history* is a **sequence** of numbers. NumPy gives us the **array**: a list of numbers we can do math on all at once. This is called **vectorization**, and it is both faster and clearer than looping.


In [ ]:
# Five days of closing prices for one stock
prices = np.array([100.0, 101.5, 99.8, 102.3, 104.0])
print("Prices:", prices)

# Daily returns: today vs yesterday, computed for ALL days at once (no loop!)
# prices[1:] means "from the 2nd element onward"; prices[:-1] means "up to the last".
daily_returns = (prices[1:] - prices[:-1]) / prices[:-1]
print("Daily returns:", np.round(daily_returns, 4))

# Array-wide summaries
print("Average daily return:", round(daily_returns.mean(), 4))
print("Highest price:", prices.max(), " Lowest price:", prices.min())

**Why this matters:** Notice we computed five days of returns in *one line* without writing a loop. When `prices` has 10 million entries (tick data for a year), this same line still works — and runs in milliseconds. Vectorized thinking is the core programming skill of a quant.


### 0.3 Functions — packaging an idea so you can reuse it

A **function** is a reusable recipe: give it inputs, it gives back an output. We'll write functions for every formula we derive, so the math becomes a tool we can call by name.


In [ ]:
def simple_return(start, end):
    """Return the simple (arithmetic) return between two prices."""
    return (end - start) / start

def log_return(start, end):
    """Return the log return. Log returns add up over time, which is why quants love them."""
    return np.log(end / start)

print("Simple return 100 -> 110:", simple_return(100, 110))
print("Log return    100 -> 110:", round(log_return(100, 110), 5))

# Why log returns? They are ADDITIVE. Two days of +10% then -10%:
print("\nSimple: a +10% day then a -10% day leaves you at",
      100 * (1.10) * (0.90), "(a LOSS, even though the numbers look symmetric)")
print("Log returns of the two days sum to:",
      round(log_return(100,110) + log_return(110,99), 5), "(matches the true log move)")

### 0.4 Your first chart

Seeing data is half of understanding it. Let's simulate and plot a price path.


In [ ]:
days = np.arange(0, 100)                       # day numbers 0,1,...,99
# A pretend price that drifts up with some wiggle
path = 100 + np.cumsum(np.random.normal(0.1, 1.0, size=100))

plt.plot(days, path, color="navy")
plt.title("A simulated 100-day price path")
plt.xlabel("Day")
plt.ylabel("Price ($)")
plt.show()

print("This 'random walk with drift' is, surprisingly, our first model of how stock prices move.")
print("We will make it rigorous in Part 4 (Stochastic Calculus).")

> **Your turn (0.5):** Change the `0.1` in the simulation to `-0.2` and re-run. What happens to the overall direction of the path? That number is the *drift* — the average daily push up or down.


---
# Part 1 — Calculus

**The trading question:** *How fast is something changing right now, and how do small changes add up?*

Calculus is the mathematics of **change** and **accumulation**. It has two halves:

- **Differentiation** answers *"how fast?"* — the slope of a curve at a point. In finance: how much does an option's price move when the stock moves? (That's the famous "delta".)
- **Integration** answers *"how much total?"* — the area under a curve, i.e. accumulation. In finance: summing up continuous interest, or the expected payoff of an option.

We'll build both from intuition, with code, and only light notation.


### 1.1 The derivative as a slope — "rise over run"

Imagine a price curve. The **derivative** at a point is the slope of the line that just *touches* the curve there (the tangent). Steeper slope = price changing faster.

The definition: take two points very close together and compute rise/run. As the gap shrinks toward zero, you get the *instantaneous* rate of change. In symbols, for a function $f(x)$:

$$f'(x) \approx \frac{f(x+h) - f(x)}{h} \quad\text{for a tiny } h.$$

Read that as: "the change in output divided by the change in input." Let's compute it numerically.


In [ ]:
def numerical_derivative(f, x, h=1e-6):
    """Approximate the slope of f at point x using a tiny step h."""
    return (f(x + h) - f(x - h)) / (2 * h)   # 'central difference' — more accurate

# Example: f(x) = x^2.  Calculus says the derivative is 2x.
f = lambda x: x**2
print("Numerical slope of x^2 at x=3:", round(numerical_derivative(f, 3), 6))
print("Exact answer (2x = 6):        ", 6)

# Example: f(x) = x^3 - 2x.  Exact derivative is 3x^2 - 2.
g = lambda x: x**3 - 2*x
print("\nNumerical slope of x^3-2x at x=2:", round(numerical_derivative(g, 2), 6))
print("Exact (3*4 - 2 = 10):            ", 10)

### 1.2 Seeing the tangent line

Let's *draw* the curve and the tangent line our derivative produces. Watch how the line kisses the curve at exactly one point with the same slope.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

f = lambda x: x**2
x0 = 1.5
slope = numerical_derivative(f, x0)

xs = np.linspace(-1, 4, 200)
tangent = f(x0) + slope * (xs - x0)   # line through (x0, f(x0)) with our slope

plt.plot(xs, f(xs), label="f(x) = x²", color="navy")
plt.plot(xs, tangent, "--", label=f"tangent at x={x0} (slope={slope:.2f})", color="crimson")
plt.scatter([x0], [f(x0)], color="black", zorder=5)
plt.legend(); plt.title("The derivative is the slope of the tangent line"); plt.show()

### 1.3 Trading application: Delta and Gamma

This is where calculus earns its keep. An **option** is a contract whose value $V$ depends on the underlying stock price $S$.

- **Delta** $= \dfrac{\partial V}{\partial S}$ — the *first derivative*: how much the option value changes for a \$1 move in the stock. If delta = 0.6, the option gains ~\$0.60 when the stock gains \$1. Traders use it to **hedge**: hold delta shares to neutralize small moves.
- **Gamma** $= \dfrac{\partial^2 V}{\partial S^2}$ — the *second derivative* (the derivative of delta): how fast delta itself changes. High gamma means your hedge goes stale quickly.

We don't need the option pricing formula yet — let's just *feel* delta and gamma on a curved payoff.


In [ ]:
# Pretend an option's value as a function of stock price looks like this smooth curve.
# (In Part 4 we'll derive the real Black-Scholes version.)
def option_value(S):
    return np.maximum(S - 100, 0) * 0.0 + 8 * np.exp(-((S-100)/15)**2) + np.maximum(S-100,0)*0.4

def first_deriv(f, x, h=1e-4):
    return (f(x+h) - f(x-h)) / (2*h)

def second_deriv(f, x, h=1e-2):
    return (f(x+h) - 2*f(x) + f(x-h)) / h**2

S_now = 100
print("Option value at S=100:", round(option_value(S_now), 3))
print("Delta (dV/dS):        ", round(first_deriv(option_value, S_now), 4),
      "-> hold this many shares to hedge")
print("Gamma (d²V/dS²):      ", round(second_deriv(option_value, S_now), 5),
      "-> how fast the hedge decays")

S = np.linspace(70, 130, 200)
plt.plot(S, option_value(S), color="navy", label="Option value V(S)")
plt.axvline(100, ls=":", color="gray")
plt.xlabel("Stock price S"); plt.ylabel("Option value V"); plt.legend()
plt.title("Delta is the slope here; Gamma is how the slope curves"); plt.show()

### 1.4 Integration — accumulating change

**Integration** is the reverse of differentiation: it adds up infinitely many tiny pieces to get a total. Geometrically it's the **area under a curve**.

Why a quant cares: the **expected payoff** of an option is the payoff at each possible price, *weighted by how likely that price is*, summed across all prices — that's an integral. Continuous compounding of interest is also an integral.

We'll approximate area the honest way: chop the region into thin rectangles and add them (a **Riemann sum**).


In [ ]:
def integral(f, a, b, n=10000):
    """Approximate the area under f from a to b using n rectangles."""
    xs = np.linspace(a, b, n)
    widths = (b - a) / n
    return np.sum(f(xs) * widths)

# The area under f(x)=x^2 from 0 to 3 is exactly 3^3/3 = 9.
print("Numerical area under x² from 0 to 3:", round(integral(lambda x: x**2, 0, 3), 4))
print("Exact (3³/3 = 9):                   ", 9)

# Continuous compounding: $1 growing at rate r for time T is e^(rT).
# It arises from integrating the growth rate. Check r=5%, T=10yr:
r, T = 0.05, 10
print("\n$1 continuously compounded at 5% for 10y:", round(np.exp(r*T), 4))

### 1.5 The Fundamental Theorem, in one breath

Differentiation and integration are **inverses** of each other — like multiply and divide. If you accumulate a rate of change and then ask how fast the accumulation grows, you get the rate back. This is the *Fundamental Theorem of Calculus*, and it's why we can move freely between "how fast" and "how much" — a move we'll make constantly when pricing derivatives.

> **Your turn (1.6):** Use `numerical_derivative` to estimate the slope of `np.sin` at `x=0`. Calculus says it should be `cos(0) = 1`. Then use `integral` to find the area under `np.sin` from `0` to `np.pi` (the exact answer is `2`).


---
# Part 2 — Linear Algebra

**The trading question:** *I don't hold one asset — I hold a whole portfolio. How do I do math on all of them at once?*

Linear algebra is the language of **many numbers at once**, arranged into **vectors** (a list) and **matrices** (a grid). A portfolio of 500 stocks, their weights, their pairwise relationships — all of it lives naturally in vectors and matrices. NumPy was *built* for this, so the code is short and the payoff is huge.


### 2.1 Vectors — a list of numbers with meaning

A **vector** is just an ordered list of numbers, e.g. the weights of each holding in your portfolio. We can add vectors, scale them, and — crucially — take their **dot product**.

The **dot product** multiplies two vectors element-by-element and sums the result. If `w` is the weight in each asset and `r` is each asset's return, then `w · r` is your **portfolio return** in a single operation.


In [ ]:
import numpy as np

# Portfolio weights: 50% in asset A, 30% in B, 20% in C (must sum to 1)
weights  = np.array([0.5, 0.3, 0.2])
returns  = np.array([0.08, -0.02, 0.05])   # this period's return of each asset

portfolio_return = weights @ returns        # '@' is the dot product in NumPy
print("Portfolio return:", round(portfolio_return * 100, 3), "%")

# Equivalent, the long way, to show what '@' actually does:
manual = sum(w * r for w, r in zip(weights, returns))
print("Same thing, computed by hand:", round(manual * 100, 3), "%")
print("Weights sum to:", weights.sum(), "(a valid portfolio uses 100% of capital)")

### 2.2 Matrices — a grid of numbers

A **matrix** is a rectangle of numbers. The most important matrix in finance is the **covariance matrix**: it stores how every asset moves *with* every other asset.

- The diagonal entries are each asset's **variance** (its own riskiness).
- The off-diagonal entries are **covariances** (do A and B move together or oppositely?).

This single object is the engine of risk management and diversification.


In [ ]:
# Simulate 1 year (252 trading days) of returns for 3 assets.
# Asset A and B are positively correlated; C moves somewhat opposite to A.
np.random.seed(7)
n_days = 252
A = np.random.normal(0.0005, 0.012, n_days)
B = 0.7 * A + np.random.normal(0.0002, 0.008, n_days)   # follows A
C = -0.4 * A + np.random.normal(0.0006, 0.015, n_days)  # leans opposite to A

R = np.column_stack([A, B, C])      # a 252 x 3 matrix of returns
cov = np.cov(R, rowvar=False)       # the 3x3 covariance matrix

print("Covariance matrix (x10,000 for readability):")
print(np.round(cov * 1e4, 2))
print("\nDiagonal = each asset's variance (own risk).")
print("Off-diagonal = how pairs co-move. Note A–C is negative (they hedge each other).")

### 2.3 Why diversification works — matrices make it obvious

Here is the single most important result in portfolio theory, and it falls right out of matrix multiplication. The **variance of a portfolio** is:

$$\sigma_p^2 = w^{\top} \, \Sigma \, w$$

Read aloud: "portfolio variance equals the weight vector, times the covariance matrix, times the weight vector again." The negative off-diagonal terms (assets that move oppositely) *subtract* from total risk — that's diversification, captured in one matrix expression.


In [ ]:
def portfolio_volatility(weights, cov_matrix):
    """Annualized portfolio volatility (std dev) from weights and a daily cov matrix."""
    daily_var = weights @ cov_matrix @ weights     # w^T Σ w
    return np.sqrt(daily_var * 252)                 # scale daily -> yearly

w_concentrated = np.array([1.0, 0.0, 0.0])          # all-in on asset A
w_diversified  = np.array([0.4, 0.2, 0.4])          # spread across A, B, C

print("Volatility, all-in on A   :", round(portfolio_volatility(w_concentrated, cov)*100, 2), "%")
print("Volatility, diversified   :", round(portfolio_volatility(w_diversified,  cov)*100, 2), "%")
print("\nDiversifying LOWERED risk — without us doing anything but spreading the weights.")
print("That free lunch is the whole point of the covariance matrix.")

### 2.4 Eigenvalues & SVD — finding hidden structure in markets

When hundreds of stocks move, they don't move independently — they share **common factors** (the whole market rises, a sector rallies, etc.). 

**Eigenvalues / eigenvectors** and the closely related **Singular Value Decomposition (SVD)** are tools that find these hidden directions of co-movement. Applied to a covariance matrix, this is called **Principal Component Analysis (PCA)**. The first principal component of stock returns is almost always "the market itself."

Let's find the dominant factor in our 3-asset returns.


In [ ]:
# PCA via eigen-decomposition of the covariance matrix.
eigvals, eigvecs = np.linalg.eigh(cov)        # eigh: for symmetric matrices like cov
# Sort from largest to smallest 'energy'
order = np.argsort(eigvals)[::-1]
eigvals, eigvecs = eigvals[order], eigvecs[:, order]

explained = eigvals / eigvals.sum()
print("Share of total variance explained by each factor:")
for i, e in enumerate(explained, 1):
    print(f"  Factor {i}: {e*100:5.1f}%")

print("\nThe top factor's direction (its eigenvector):", np.round(eigvecs[:, 0], 3))
print("This is the dominant 'market move' — the single pattern that explains the most co-movement.")

### 2.5 Solving systems — linear regression preview

Linear algebra also lets us *solve* for unknowns. "Find the weights `x` such that `M x = y`" is solved with `np.linalg.solve`. This same machinery powers **linear regression** (Part 3), where we solve for the line that best fits market data — e.g. a stock's *beta* to the market.

> **Your turn (2.6):** Change asset C's construction so it *follows* A instead of opposing it (use `+0.4 * A` instead of `-0.4 * A`), re-run 2.2–2.3, and watch the diversified portfolio's volatility *rise*. Negative correlation is what makes diversification powerful.


---
# Part 3 — Probability & Statistics

**The trading question:** *What is my expected profit, and how uncertain is that profit?*

This is arguably the heart of quant finance. Markets are uncertain, so we describe them with **probability** (the math of chance) and measure them with **statistics** (extracting truth from data). Every trade is a bet, and a quant's edge comes from understanding the *distribution* of outcomes, not just the average.


### 3.1 Expected value — the average outcome

The **expected value** (EV) is the long-run average if you could repeat a bet infinitely. You compute it by weighting each outcome by its probability and summing:

$$E[X] = \sum_i p_i \, x_i$$

Read aloud: "the sum of each outcome times its probability." A trade with **positive expected value** makes money on average. This is the first question a quant asks of any strategy.


In [ ]:
# A classic example: 60% chance to win $100, 40% chance to lose $80.
outcomes      = np.array([100, -80])
probabilities = np.array([0.60, 0.40])

EV = np.sum(outcomes * probabilities)
print("Expected value of the bet: $", round(EV, 2))
print("Positive EV -> on average, this bet makes money. But average isn't the whole story...")

# Verify by simulation: play the bet 1,000,000 times and average the result.
import numpy as np
np.random.seed(1)
plays = np.random.choice(outcomes, size=1_000_000, p=probabilities)
print("Simulated average over 1,000,000 plays: $", round(plays.mean(), 3))

### 3.2 Variance & standard deviation — quantifying risk

Two strategies can share the *same* expected value but feel completely different. **Variance** measures how spread out the outcomes are; its square root, the **standard deviation** (often called **volatility** in finance, symbol $\sigma$), is in the same units as the outcome and is the industry's main risk number.

$$\text{Var}(X) = E\big[(X - E[X])^2\big], \qquad \sigma = \sqrt{\text{Var}(X)}$$

Read aloud: "the average squared distance from the mean." Big spread = big risk.


In [ ]:
# Same bet as before. Compute its variance and standard deviation.
mean = EV
var = np.sum(probabilities * (outcomes - mean)**2)
std = np.sqrt(var)
print("Expected value : $", round(mean, 2))
print("Std deviation  : $", round(std, 2))
print("\nThe edge is only $28 but the swing is ~$88 — a bumpy ride for a small edge.")
print("This ratio of edge-to-risk is what professionals actually care about (see Sharpe ratio below).")

### 3.3 The Sharpe ratio — reward per unit of risk

Quants rarely chase raw return; they chase **risk-adjusted return**. The **Sharpe ratio** divides excess return by volatility:

$$\text{Sharpe} = \frac{\text{average return} - \text{risk-free rate}}{\text{volatility}}$$

A higher Sharpe means more reward for each unit of stomach-churning risk. It's the single number most used to compare strategies.


In [ ]:
def sharpe_ratio(returns, risk_free_daily=0.0):
    """Annualized Sharpe ratio from a series of daily returns."""
    excess = returns - risk_free_daily
    return np.sqrt(252) * excess.mean() / excess.std()

np.random.seed(3)
strat_smooth = np.random.normal(0.0006, 0.008, 252)   # modest return, low vol
strat_wild   = np.random.normal(0.0012, 0.030, 252)   # double return, but 4x the vol

print("Smooth strategy: return", round(strat_smooth.mean()*252*100,1), "% | Sharpe",
      round(sharpe_ratio(strat_smooth), 2))
print("Wild strategy  : return", round(strat_wild.mean()*252*100,1), "% | Sharpe",
      round(sharpe_ratio(strat_wild), 2))
print("\nThe 'wild' strategy may earn more, but per unit of risk the smooth one is often better.")

### 3.4 Distributions — the shape of randomness

A **probability distribution** describes how likely each outcome is. The famous **Normal (Gaussian)** distribution — the bell curve — is the default model for returns because of the **Central Limit Theorem**: sums of many small independent shocks tend toward a bell curve.

But beware: real market returns have **fat tails** — extreme crashes happen far more often than a normal curve predicts. We'll show both.


In [ ]:
import matplotlib.pyplot as plt
from scipy import stats

x = np.linspace(-5, 5, 400)
normal_pdf = stats.norm.pdf(x)                 # the bell curve
fat_tailed = stats.t.pdf(x, df=3)              # Student-t: same center, heavier tails

plt.plot(x, normal_pdf, label="Normal (textbook assumption)", color="navy")
plt.plot(x, fat_tailed, label="Fat-tailed (closer to real markets)", color="crimson")
plt.title("Why 'once-in-a-century' crashes happen every few years")
plt.xlabel("Return (in std devs)"); plt.ylabel("Probability density"); plt.legend()
plt.show()

# Quantify the tail: probability of a move beyond -3 std devs ('a crash')
print("Normal model says a <-3σ day has probability:", round(stats.norm.cdf(-3)*100, 3), "%")
print("Fat-tailed model says it's:                  ", round(stats.t.cdf(-3, df=3)*100, 3), "%")
print("The fat-tailed world is several times more dangerous in the tails.")

### 3.5 Value at Risk (VaR) — a number for "how bad can it get?"

Risk desks summarize downside with **Value at Risk**: the loss you won't exceed on, say, 95% of days. It's a **percentile** (quantile) of the return distribution — pure statistics, used everywhere from hedge funds to bank regulation.


In [ ]:
np.random.seed(11)
daily_pnl = np.random.normal(0.0005, 0.015, 100_000)   # simulated daily returns

VaR_95 = np.percentile(daily_pnl, 5)     # the 5th percentile = worst 5% threshold
VaR_99 = np.percentile(daily_pnl, 1)
print("95% 1-day VaR:", round(VaR_95*100, 2), "% (we expect to do worse than this ~1 day in 20)")
print("99% 1-day VaR:", round(VaR_99*100, 2), "% (~1 day in 100)")
print("\nOn a $1,000,000 book, the 99% VaR is about $", round(abs(VaR_99)*1_000_000, 0))

### 3.6 Linear regression — measuring a stock's "beta"

**Regression** finds the best straight line through a cloud of data. In finance, regressing a stock's returns on the market's returns gives **beta** — how sensitive the stock is to the overall market. Beta = 1.5 means the stock tends to move 1.5x as much as the market. This connects directly back to Part 2's linear algebra (we're solving a least-squares system).


In [ ]:
np.random.seed(21)
market = np.random.normal(0.0004, 0.01, 500)            # market returns
true_beta = 1.4
stock = true_beta * market + np.random.normal(0, 0.006, 500)   # stock follows market + noise

# Fit a line: stock = alpha + beta * market.  polyfit solves the least-squares problem.
beta, alpha = np.polyfit(market, stock, 1)
print("Estimated beta :", round(beta, 3), "(true value was 1.4)")
print("Estimated alpha:", round(alpha, 5), "(the part of return NOT explained by the market)")

plt.scatter(market*100, stock*100, s=8, alpha=0.4, label="daily observations")
xs = np.linspace(market.min(), market.max(), 50)
plt.plot(xs*100, (alpha + beta*xs)*100, color="crimson", lw=2, label=f"fit: beta={beta:.2f}")
plt.xlabel("Market return (%)"); plt.ylabel("Stock return (%)"); plt.legend()
plt.title("Regression recovers the stock's sensitivity to the market"); plt.show()

> **Your turn (3.7):** Build your own bet. Define `outcomes = [50, -10, -100]` with `probabilities = [0.5, 0.3, 0.2]`. Compute its expected value and standard deviation using the formulas above. Is it a bet you'd take? Then estimate its risk with a simulation of 1,000,000 plays.


---
# Part 4 — Stochastic Calculus

**The trading question:** *Prices wiggle randomly through time. How do we describe that motion mathematically — and use it to price an option?*

This is the capstone of quant mathematics. "Stochastic" just means **random**. Stochastic calculus is *calculus for random, jiggling paths* — the kind that stock prices trace out. It sounds intimidating, but we'll build it one honest step at a time: random walk → Brownian motion → geometric Brownian motion → Itô's idea → Black–Scholes. You already have every prerequisite from Parts 1–3.


### 4.1 The random walk — randomness accumulating over time

Start at 0. Each step, flip a coin: +1 for heads, −1 for tails. Your position over time is a **random walk**. It is the simplest model of an unpredictable, accumulating process — and the seed of everything that follows.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(5)
n_steps = 500
for _ in range(8):                                   # draw 8 different random walks
    steps = np.random.choice([-1, 1], size=n_steps)  # coin flips
    walk = np.cumsum(steps)                           # running total = the path
    plt.plot(walk, alpha=0.7)

plt.title("Eight random walks — same rules, wildly different paths")
plt.xlabel("Step"); plt.ylabel("Position"); plt.show()
print("No path is predictable, yet as a GROUP they spread out in a regular, measurable way.")
print("That regularity (the spread grows with the square root of time) is the key insight.")

### 4.2 Brownian motion — the random walk in continuous time

Shrink the time between steps toward zero and the random walk becomes **Brownian motion** (also called a **Wiener process**, symbol $W_t$). It is the continuous-time idealization of "pure randomness accumulating." Two facts define it and we'll verify both in code:

1. Over a time interval of length $t$, the change is **Normally distributed** with variance $t$ — so its spread (std dev) grows like $\sqrt{t}$.
2. Non-overlapping pieces are **independent**.


In [ ]:
def brownian_motion(T=1.0, n=1000, paths=1):
    """Simulate Brownian motion on [0,T] using n time steps."""
    dt = T / n
    # Each increment is Normal(0, dt). Cumulative sum builds the path.
    increments = np.random.normal(0, np.sqrt(dt), size=(paths, n))
    W = np.cumsum(increments, axis=1)
    W = np.hstack([np.zeros((paths, 1)), W])   # start every path at 0
    t = np.linspace(0, T, n + 1)
    return t, W

np.random.seed(0)
t, W = brownian_motion(T=1.0, n=1000, paths=5)
for i in range(5):
    plt.plot(t, W[i], alpha=0.8)
plt.title("Brownian motion: the continuous-time limit of a random walk")
plt.xlabel("Time t"); plt.ylabel("W(t)"); plt.show()

# Verify 'spread grows like sqrt(t)': simulate many paths, check std at several times.
_, Wm = brownian_motion(T=1.0, n=1000, paths=20000)
for frac in [0.25, 0.5, 1.0]:
    idx = int(frac * 1000)
    print(f"At t={frac}: empirical std = {Wm[:, idx].std():.3f},  theory √t = {np.sqrt(frac):.3f}")

### 4.3 Geometric Brownian Motion — the standard stock-price model

Stock prices can't go negative, and they move in *percentage* terms (a \$1 move means more to a \$10 stock than a \$1000 stock). So we don't model the price directly as Brownian motion — we model its **growth rate**. The result is **Geometric Brownian Motion (GBM)**, the model underlying Black–Scholes:

$$S_t = S_0 \, \exp\!\Big( \big(\mu - \tfrac{1}{2}\sigma^2\big)\,t \;+\; \sigma\, W_t \Big)$$

In words: the price equals the starting price times $e$ raised to *(a steady drift term) plus (a random Brownian term scaled by volatility $\sigma$)*. Here $\mu$ (mu) is the average growth rate and $\sigma$ (sigma) is the volatility from Part 3.


In [ ]:
def gbm_paths(S0=100, mu=0.08, sigma=0.2, T=1.0, n=252, paths=10):
    """Simulate Geometric Brownian Motion stock-price paths."""
    dt = T / n
    t, W = brownian_motion(T=T, n=n, paths=paths)
    drift = (mu - 0.5 * sigma**2) * t                # the steady part
    S = S0 * np.exp(drift + sigma * W)               # GBM formula, vectorized
    return t, S

np.random.seed(2)
t, S = gbm_paths(S0=100, mu=0.08, sigma=0.2, paths=30)
plt.plot(t, S.T, alpha=0.5)
plt.title("30 simulated 1-year stock paths (start $100, 8% drift, 20% vol)")
plt.xlabel("Time (years)"); plt.ylabel("Price ($)"); plt.show()

print("Average ending price:", round(S[:, -1].mean(), 2),
      " (≈ 100·e^(0.08) =", round(100*np.exp(0.08), 2), ", the expected growth)")

### 4.4 Itô's insight — why random calculus is different

Here's the one genuinely new idea. In ordinary calculus from Part 1, if you take a tiny step `dt`, then `(dt)²` is so small you ignore it. But Brownian motion is so jagged that the *square* of its tiny random step, `(dW)²`, does **not** vanish — it behaves like `dt` itself.

This single fact (`(dW)² ≈ dt`) is the engine of **Itô's Lemma**, the chain rule for random functions. It's the reason GBM has that mysterious `−½σ²` correction term in the exponent. Let's verify the fact numerically — no proof needed, just see it.


In [ ]:
# Claim: summing (dW)^2 over [0,1] gives ≈ 1 (= the length of the interval),
# NOT ≈ 0 as ordinary calculus intuition would suggest.
np.random.seed(8)
n = 100_000
dt = 1.0 / n
dW = np.random.normal(0, np.sqrt(dt), n)

print("Sum of (dW)²  over [0,1]:", round(np.sum(dW**2), 4), " <- converges to 1 = T")
print("Sum of (dt)²  over [0,1]:", round(n * dt**2, 8),     " <- vanishes, as ordinary calc expects")
print("\nSo (dW)² behaves like dt. THIS is what makes stochastic calculus its own subject,")
print("and it forces the -½σ² term that keeps option pricing honest.")

### 4.5 The payoff: pricing an option two ways

An option to **buy** a stock at a fixed **strike** price K (a *call option*) pays `max(S_T − K, 0)` at expiry: if the stock ends above K you pocket the difference, otherwise it expires worthless.

We'll price it **two independent ways** and watch them agree:
1. **Monte Carlo** — simulate thousands of GBM paths, average the payoff, discount to today. (Uses Parts 3 & 4.)
2. **Black–Scholes formula** — the celebrated closed-form solution that won a Nobel Prize, which integration (Part 1) and the normal distribution (Part 3) make possible.


In [ ]:
from scipy import stats

def black_scholes_call(S0, K, r, sigma, T):
    """Exact Black-Scholes price of a European call option."""
    d1 = (np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S0*stats.norm.cdf(d1) - K*np.exp(-r*T)*stats.norm.cdf(d2)

def monte_carlo_call(S0, K, r, sigma, T, n_paths=200_000):
    """Price the same call by simulating terminal stock prices under GBM."""
    Z = np.random.normal(0, 1, n_paths)
    # Under risk-neutral pricing we use drift = r (the risk-free rate).
    ST = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    payoffs = np.maximum(ST - K, 0)
    return np.exp(-r*T) * payoffs.mean()    # average payoff, discounted to today

np.random.seed(123)
S0, K, r, sigma, T = 100, 105, 0.03, 0.2, 1.0
bs = black_scholes_call(S0, K, r, sigma, T)
mc = monte_carlo_call(S0, K, r, sigma, T)

print(f"Black–Scholes formula price : ${bs:.4f}")
print(f"Monte Carlo simulation price: ${mc:.4f}")
print(f"Difference                  : ${abs(bs-mc):.4f}  (tiny — they agree!)")
print("\nTwo completely different methods, one answer. That's the payoff of everything in Parts 1–4.")

### 4.6 Connecting back to calculus: the Greeks live here

Remember **delta** and **gamma** from Part 1? With the Black–Scholes formula in hand, they're just derivatives of the price with respect to the stock. Let's compute delta numerically and confirm it matches the formula's known answer, `N(d1)`.


In [ ]:
def bs_delta_numerical(S0, K, r, sigma, T, h=1e-3):
    up   = black_scholes_call(S0 + h, K, r, sigma, T)
    down = black_scholes_call(S0 - h, K, r, sigma, T)
    return (up - down) / (2*h)

d1 = (np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
print("Delta (numerical derivative of price):", round(bs_delta_numerical(S0,K,r,sigma,T), 4))
print("Delta (exact formula N(d1)):          ", round(stats.norm.cdf(d1), 4))
print("\nCalculus (Part 1) + probability (Part 3) + GBM (Part 4) = a hedge ratio you can trade on.")

> **Your turn (4.7):** Raise volatility `sigma` from `0.2` to `0.4` and re-price the option both ways. Does the call get more or less valuable? (Hint: more uncertainty means more upside potential, while the downside is capped at zero — this is *why options traders care so much about volatility*.)


---
# Part 5 — Capstone Projects

You now have all five pillars: programming, calculus, linear algebra, probability & statistics, and stochastic calculus. These three projects each weave several pillars together into something a working quant actually builds. Treat them as templates you can extend for a portfolio.


## Project 1 — Monte Carlo Strategy Simulator

**Pillars used:** Probability (Part 3) + Stochastic processes (Part 4) + Programming (Part 0).

A Monte Carlo simulator answers "what's the *distribution* of outcomes for my strategy?" by replaying it across thousands of randomly simulated futures. We'll evaluate a simple buy-and-hold position and report expected return, volatility, the chance of a loss, and the worst-case (VaR).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simulate_strategy(S0=100, mu=0.07, sigma=0.2, T=1.0, n_sims=50_000):
    """Simulate terminal value of $S0 buy-and-hold under GBM across many futures."""
    Z = np.random.normal(0, 1, n_sims)
    ST = S0 * np.exp((mu - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    returns = (ST - S0) / S0
    return returns

np.random.seed(2024)
rets = simulate_strategy()

print("=== Monte Carlo report: 1-year buy & hold ===")
print(f"Expected return     : {rets.mean()*100:6.2f}%")
print(f"Volatility (std)    : {rets.std()*100:6.2f}%")
print(f"Probability of loss : {(rets < 0).mean()*100:6.2f}%")
print(f"95% VaR (worst 5%)  : {np.percentile(rets, 5)*100:6.2f}%")
print(f"Best 5% upside      : {np.percentile(rets, 95)*100:6.2f}%")

plt.hist(rets*100, bins=80, color="steelblue", edgecolor="white")
plt.axvline(rets.mean()*100, color="black", ls="--", label="expected")
plt.axvline(np.percentile(rets,5)*100, color="crimson", ls="--", label="95% VaR")
plt.title("Distribution of 1-year outcomes (Monte Carlo)")
plt.xlabel("Return (%)"); plt.ylabel("Frequency"); plt.legend(); plt.show()

## Project 2 — Portfolio Optimizer (Markowitz)

**Pillars used:** Linear algebra (Part 2) + Probability/statistics (Part 3) + Programming.

Given several assets, what mix maximizes return *per unit of risk* (Sharpe ratio)? Harry Markowitz won a Nobel Prize for framing this. We'll randomly sample thousands of portfolios, plot the **efficient frontier**, and find the best Sharpe portfolio — all powered by the covariance matrix from Part 2.


In [ ]:
np.random.seed(55)
# Three assets with chosen annual returns, vols, and a correlation structure.
mean_returns = np.array([0.10, 0.14, 0.07])           # expected annual returns
vols         = np.array([0.18, 0.28, 0.12])           # annual volatilities
corr = np.array([[1.0, 0.3, -0.1],
                 [0.3, 1.0,  0.2],
                 [-0.1,0.2,  1.0]])
cov = np.outer(vols, vols) * corr                     # covariance matrix Σ

n_port = 40_000
results = np.zeros((n_port, 3))                       # store return, vol, sharpe
weights_store = []
rf = 0.03                                             # risk-free rate

for i in range(n_port):
    w = np.random.random(3); w /= w.sum()             # random weights summing to 1
    port_ret = w @ mean_returns                       # Part 2: dot product
    port_vol = np.sqrt(w @ cov @ w)                   # Part 2: w^T Σ w
    sharpe   = (port_ret - rf) / port_vol             # Part 3: Sharpe ratio
    results[i] = [port_ret, port_vol, sharpe]
    weights_store.append(w)

best = results[:, 2].argmax()
bw = weights_store[best]
print("=== Optimal (max-Sharpe) portfolio ===")
print(f"Weights  : A={bw[0]*100:.1f}%  B={bw[1]*100:.1f}%  C={bw[2]*100:.1f}%")
print(f"Return   : {results[best,0]*100:.2f}%")
print(f"Volatility: {results[best,1]*100:.2f}%")
print(f"Sharpe   : {results[best,2]:.3f}")

sc = plt.scatter(results[:,1]*100, results[:,0]*100, c=results[:,2], cmap="viridis", s=5)
plt.colorbar(sc, label="Sharpe ratio")
plt.scatter(results[best,1]*100, results[best,0]*100, color="red", marker="*", s=300, label="Best Sharpe")
plt.xlabel("Volatility (%)"); plt.ylabel("Expected return (%)")
plt.title("The Efficient Frontier — each dot is a portfolio"); plt.legend(); plt.show()

## Project 3 — Implied Volatility Surface

**Pillars used:** Calculus root-finding (Part 1) + Stochastic/Black–Scholes (Part 4) + Programming.

The Black–Scholes formula turns volatility into a price. **Implied volatility** runs it *backwards*: given a market price, what volatility does it imply? We can't solve this with algebra, so we use a **numerical root-finder** (a calculus technique). Plotting implied vol across strikes and expiries gives the **volatility surface** — the central object on every options desk. Real surfaces show a "smile/skew," which is the market admitting that returns have the fat tails we met in Part 3.


In [ ]:
from scipy import stats
from scipy.optimize import brentq

def black_scholes_call(S0, K, r, sigma, T):
    d1 = (np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S0*stats.norm.cdf(d1) - K*np.exp(-r*T)*stats.norm.cdf(d2)

def implied_vol(market_price, S0, K, r, T):
    """Find the sigma that makes Black-Scholes match the market price (root-finding)."""
    objective = lambda sig: black_scholes_call(S0, K, r, sig, T) - market_price
    try:
        return brentq(objective, 1e-4, 5.0)          # search for the crossing point
    except ValueError:
        return np.nan

S0, r = 100, 0.03
strikes = np.linspace(80, 120, 17)
expiries = np.array([0.25, 0.5, 1.0, 2.0])

# Build synthetic market prices that embed a 'volatility smile' (higher vol away from the money).
surface = np.zeros((len(expiries), len(strikes)))
for i, T in enumerate(expiries):
    for j, K in enumerate(strikes):
        smile_vol = 0.20 + 0.0008*(K-100)**2/ (1+T)   # higher for deep ITM/OTM strikes
        mkt = black_scholes_call(S0, K, r, smile_vol, T)
        surface[i, j] = implied_vol(mkt, S0, K, r, T)  # recover it via root-finding

for i, T in enumerate(expiries):
    plt.plot(strikes, surface[i]*100, marker="o", label=f"T = {T}y")
plt.axvline(100, ls=":", color="gray", label="at-the-money")
plt.title("Implied Volatility Surface (the 'smile')")
plt.xlabel("Strike price K"); plt.ylabel("Implied volatility (%)"); plt.legend(); plt.show()
print("The U-shape is the market pricing in fat tails — exactly the risk Part 3 warned about.")

---
# Where to go next

You've now built, with your own code, the core toolkit of a quantitative analyst. Each pillar reinforces the others:

- **Programming** lets you express any idea as repeatable computation.
- **Calculus** measures change and accumulation — delta, gamma, continuous compounding.
- **Linear algebra** scales you from one asset to whole portfolios — covariance, PCA, optimization.
- **Probability & statistics** quantify expected reward and risk — EV, volatility, Sharpe, VaR, regression.
- **Stochastic calculus** models random price paths and prices derivatives — Brownian motion, GBM, Black–Scholes.

**Suggested next steps to deepen this into a portfolio:**

1. **Use real data.** Swap the simulated prices for real history (e.g. via `yfinance`) and recompute betas, covariance matrices, and Sharpe ratios on actual stocks.
2. **Backtest a strategy.** Build a simple moving-average crossover signal and Monte-Carlo its outcomes.
3. **Extend the Greeks.** Compute vega, theta, and rho by differentiating Black–Scholes, and build a delta-hedging simulation.
4. **Study deeper.** Continue with structured roadmaps like [QuantFrame](https://quantframe.io), and classic texts: Hull's *Options, Futures, and Other Derivatives* and Shreve's *Stochastic Calculus for Finance*.

Every cell here is yours to modify. The fastest way to learn is to change a number, predict what will happen, then run it and find out. Happy quanting.
